# Public Document Metadata Analysis

Analyze synthetic metadata fields that resemble openly published documents.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Group related production patterns and identify metadata-hygiene issues without exposing personal identifiers.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
document_count = 80
documents = pd.DataFrame({
    "document_id": [f"doc-{index:03d}" for index in range(document_count)],
    "creator_tool": rng.choice(["Writer-A", "Writer-B", "PDF-Engine-C", "Scanner-D"], document_count),
    "template_id": rng.choice([f"template-{index:02d}" for index in range(10)], document_count),
    "timezone_offset": rng.choice([-8, -5, 0, 1, 5, 8], document_count),
    "revision_count": rng.integers(1, 24, document_count),
    "author_field_present": rng.binomial(1, 0.35, document_count),
})
print(documents.head(7).to_string(index=False))


document_id creator_tool template_id  timezone_offset  revision_count  author_field_present
    doc-000 PDF-Engine-C template-01                5               8                     1
    doc-001     Writer-A template-01                5              15                     0
    doc-002    Scanner-D template-04                8              10                     0
    doc-003 PDF-Engine-C template-06                0               3                     1
    doc-004 PDF-Engine-C template-04                0               4                     0
    doc-005 PDF-Engine-C template-03                5              13                     0
    doc-006    Scanner-D template-04                0              21                     0


### 2. Analyze and rank the observations


In [3]:
production_patterns = (
    documents.groupby(["creator_tool", "template_id"], as_index=False)
    .agg(documents=("document_id", "count"), timezones=("timezone_offset", "nunique"), author_fields=("author_field_present", "sum"), median_revisions=("revision_count", "median"))
)
production_patterns["metadata_hygiene_score"] = (
    1 - np.minimum(production_patterns["author_fields"] / production_patterns["documents"], 1)
).round(3)
ranked_patterns = production_patterns.sort_values(["documents", "metadata_hygiene_score"], ascending=[False, True])
print(ranked_patterns.head(10).to_string(index=False))


creator_tool template_id  documents  timezones  author_fields  median_revisions  metadata_hygiene_score
PDF-Engine-C template-05          5          3              2              17.0                   0.600
    Writer-B template-08          5          4              2              10.0                   0.600
    Writer-A template-05          4          4              3               7.0                   0.250
    Writer-A template-09          4          2              2               5.5                   0.500
   Scanner-D template-03          4          3              1               8.5                   0.750
    Writer-B template-01          4          4              1               5.0                   0.750
    Writer-B template-07          4          3              1              12.0                   0.750
PDF-Engine-C template-06          3          3              2               3.0                   0.333
PDF-Engine-C template-07          3          3              2   

## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert documents["document_id"].is_unique
assert ranked_patterns["documents"].sum() == document_count
assert ranked_patterns["metadata_hygiene_score"].between(0, 1).all()
print("Checks passed; metadata patterns do not establish authorship.")


Checks passed; metadata patterns do not establish authorship.


## Next Steps

- Hash or redact direct identifiers before analysis.
- Record document provenance and publication timestamps.
